# Retrieval evaluation
Query stored papers through the REST API and inspect their nearest neighbours and matching passages.

Start the API from the repository root with `uvicorn paper_clustering.api:app --host 127.0.0.1 --port 8000`, using your usual database configuration. Set the URL and arXiv IDs below, then run all cells. IDs must match those stored in the database; the initial examples come from `test_ids.txt`.

The API performs retrieval and reranking. Results retain its paper and passage order. **Similarity is the original vector similarity, not the reranker score**, so it need not decrease with rank. Any self-matches returned by the API are shown and labelled. An empty response can also mean the query has no stored passages. This notebook supports qualitative inspection; it does not compute relevance metrics.

In [ ]:
import os
from html import escape
from time import perf_counter
from urllib.parse import quote

import requests
from IPython.display import HTML, display

API_BASE_URL = os.environ.get("PAPER_API_URL", "http://127.0.0.1:8000").rstrip("/")
ARXIV_IDS = [
    "1206.3160",
    "1711.03860",
    "1301.1873",
]
K = 10  # Number of papers requested per query; the API accepts 1–100.
TIMEOUT_SECONDS = 120  # Allow time for server-side reranking.

## Fetch results
One `GET /query` request per distinct ID, with `arxiv_id` and `k` as query parameters. Failures are recorded without stopping the other queries. Rerun this cell to refresh results; rerunning the display cell does not call the API.

In [ ]:
if not isinstance(K, int) or isinstance(K, bool) or not 1 <= K <= 100:
    raise ValueError("K must be an integer between 1 and 100.")
if not isinstance(ARXIV_IDS, list) or not ARXIV_IDS or any(
    not isinstance(arxiv_id, str) or not arxiv_id.strip() for arxiv_id in ARXIV_IDS
):
    raise ValueError("ARXIV_IDS must be a non-empty list of non-blank strings.")

results = {}
with requests.Session() as session:
    for arxiv_id in dict.fromkeys(arxiv_id.strip() for arxiv_id in ARXIV_IDS):
        started = perf_counter()
        try:
            response = session.get(
                f"{API_BASE_URL}/query",
                params={"arxiv_id": arxiv_id, "k": K},
                timeout=TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            matches = response.json()
            if not isinstance(matches, dict) or any(
                not isinstance(pairs, list) for pairs in matches.values()
            ):
                raise ValueError("Expected a JSON object mapping paper IDs to passage lists.")
            results[arxiv_id] = {"matches": matches, "error": None}
        except (requests.RequestException, ValueError) as error:
            results[arxiv_id] = {"matches": {}, "error": str(error)}
        results[arxiv_id]["seconds"] = perf_counter() - started
        status = results[arxiv_id]["error"] or f"{len(results[arxiv_id]['matches'])} papers"
        print(f"{arxiv_id}: {status} ({results[arxiv_id]['seconds']:.1f}s)")

## Inspect neighbours
Each query has a compact summary followed by expandable passage comparisons. The summary reports the **maximum vector similarity** among each paper's returned pairs, independently of its API rank. Full passages appear side by side in API order.

In [ ]:
def paper_link(arxiv_id):
    url = "https://arxiv.org/abs/" + quote(arxiv_id, safe="/")
    return f'<a href="{escape(url, quote=True)}" target="_blank" rel="noopener noreferrer">{escape(arxiv_id)}</a>'


sections = ["""
<style>
.retrieval-eval { max-width: 1200px; line-height: 1.5; }
.retrieval-eval section { margin: 1.5em 0; border-top: 1px solid #aaa; }
.retrieval-eval table { width: 100%; border-collapse: collapse; margin: 0.8em 0; }
.retrieval-eval th, .retrieval-eval td { padding: 0.6em; text-align: left; vertical-align: top; border-bottom: 1px solid #ccc; }
.retrieval-eval .passages { table-layout: fixed; }
.retrieval-eval .passages td { white-space: pre-wrap; overflow-wrap: anywhere; }
.retrieval-eval details { margin: 0.7em 0; }
.retrieval-eval summary { cursor: pointer; }
</style>
<div class="retrieval-eval">
"""]
for arxiv_id, result in results.items():
    sections.append(f"<section><h2>Query: {paper_link(arxiv_id)}</h2>")
    sections.append(f"<p>Requested {K} neighbours · {result['seconds']:.1f}s</p>")
    if result["error"]:
        sections.append(f"<p><strong>Request failed:</strong> {escape(result['error'])}</p>")
    elif not result["matches"]:
        sections.append("<p>No matches returned. The query may have no stored passages.</p>")
    else:
        rows = []
        details = []
        for rank, (target_id, pairs) in enumerate(result["matches"].items(), start=1):
            label = paper_link(target_id)
            if target_id == arxiv_id:
                label += " (self-match)"
            best_similarity = max((pair["similarity"] for pair in pairs), default=None)
            score = f"{best_similarity:.4f}" if best_similarity is not None else "—"
            rows.append(f"<tr><td>{rank}</td><td>{label}</td><td>{score}</td><td>{len(pairs)}</td></tr>")
            comparisons = []
            for pair_number, pair in enumerate(pairs, start=1):
                comparisons.append(
                    f"<h4>Pair {pair_number} · Vector similarity: {pair['similarity']:.4f}</h4>"
                    '<table class="passages"><thead><tr><th>Query passage</th><th>Neighbour passage</th></tr></thead>'
                    f"<tbody><tr><td>{escape(pair['query_passage'])}</td>"
                    f"<td>{escape(pair['target_passage'])}</td></tr></tbody></table>"
                )
            details.append(
                f"<details><summary>#{rank} · {label} · {len(pairs)} passage pairs</summary>"
                + ("".join(comparisons) or "<p>No passage pairs returned.</p>")
                + "</details>"
            )
        sections.append(
            "<table><thead><tr><th>API rank</th><th>Neighbour</th>"
            "<th>Max vector similarity</th><th>Passage pairs</th></tr></thead><tbody>"
            + "".join(rows) + "</tbody></table>" + "".join(details)
        )
    sections.append("</section>")
sections.append("</div>")
display(HTML("".join(sections)))